# Suite CPD — The C Preprocessor dialect

`Expressions.C.Preprocessor` models the preprocessing directives C and C++ share, each standard defining its own
(ISO/IEC 9899, 6.10; ISO/IEC 14882, [cpp]), `defined(name)` within their conditions, and the comments kept among them,
as mbse-programs has them: one set of directives for both languages, the C family's dialect that C and C++ read. A
condition is a C or a C++ term, `defined(name)` one of C's or C++'s; a directive and a comment are an enumeration's items
in C Types' and Cpp Types' terms. Its kinds are validated, bind nothing but a conditional directive's macros' names
within its condition, cover C's and C++'s editions, print as mbse-programs prints them, and are refused by evaluation by
name.

In [ ]:
from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.C import Expressions as CFamily, Preprocessor as CPreprocessor, Types as CTypes
from mbse.Expressions.Cpp import Expressions as CppFamily, Types as CppTypes
from mbse.Expressions.SystemV import AMD64
from mbse.Expressions.C.Expressions import Evaluators as CV, Expressions as C, Text as CText
from mbse.Expressions.C.Preprocessor import Expressions as PP, Text as PPText
from mbse.Expressions.C.Types import Expressions as CT, Text as CTText
from mbse.Expressions.Cpp.Expressions import Evaluators as XV, Expressions as CPP, Text as XText
from mbse.Expressions.Cpp.Types import Expressions as XT
from mbse.Expressions.Framework import Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators
from support import raises

store = Builders.Store()  # the C family's dialects, the preprocessor's install before the others': in any order
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, CPreprocessor.install, CFamily.install, CTypes.install,
                CppFamily.install, CppTypes.install, AMD64.install]:
    store.transaction(install)
for define in [CFamily.define, CppFamily.define, AMD64.define, CPreprocessor.define]:  # C's and C++'s types, its editions
    store.transaction(define)
AMD64_ = store.Instances.Expressions.SystemV.AMD64  # the ABI a holder names, whose types C and C++ leave to it


def term(spec, dialect=PP.DIALECT):
    """The term value `spec` writes, the root of `dialect`'s holder, naming x86-64's ABI, written in a transaction."""
    with store.transaction():
        path = dialect.path.split(".")
        holder = getattr(getattr(store.Expressions, path[0]), path[1]).Expression
        return F.term(holder(lambda e: e.root(dialect.spec(spec)).abi([AMD64_])))


def validate(spec, bound=(), dialect=PP.DIALECT):
    return dialect.validate(term(spec, dialect), bound)


text = lambda spec, dialect=PP.DIALECT: PPText.ToText(term(spec, dialect))  # noqa: E731
union = lambda name: [b.name for b in store.registered(name).branches]  # noqa: E731: a union's branches
schemas = lambda name: {b.name: b.schema for b in store.registered(name).branches}  # noqa: E731
spelled = lambda word: lambda i: i.spelling(word)  # noqa: E731: an identifier, of its spelling
own = lambda spec: PP.DIALECT.spec(spec)  # noqa: E731: a term of its own, where any dialect's may stand
defined = lambda word, dialect=C.DIALECT: dialect.spec(lambda u: u.definedexpression(lambda d: d.name(spelled(word))))  # noqa: E731
enumerator = lambda word, dialect=CT.DIALECT: dialect.spec(lambda u: u.enumerator(lambda r: r.name(spelled(word))))  # noqa: E731
comment = lambda words, block=False: lambda u: u.comment(lambda c: c.text(words).block(block))  # noqa: E731
condition = C.binary("&&", defined("X"), C.binary(">", C.identifier("X"), 1))  # defined(X) && X > 1, a C condition
else_ = lambda items: lambda a: a.elsedirective(lambda e: e.items(items))  # noqa: E731
if_ = lambda u: u.ifdirective(lambda d: d.condition(condition).items([enumerator("A")]).alternative(  # noqa: E731
    lambda a: a.elifdefdirective(lambda e: e.negated(True).name(spelled("Z")).items([enumerator("B")]).alternative(
        else_([own(comment(" none"))])))))


<a id="CPD-01"></a>
## CPD-01 · Its union and holder: its kinds and invalid syntax, definedexpression in C's and C++'s terms, the directives and comment in C Types' and Cpp Types'; a holder refining the framework's, round-tripped through JSON

In [ ]:
assert PP.TAGS == ("includedirective", "definedirective", "ifdirective", "ifdefdirective", "elifdirective", "elifdefdirective",
                   "elsedirective", "otherdirective", "definedexpression", "comment") and PP.DIRECTIVES == PP.TAGS[:8]
assert union(PP.DIALECT.union) == [*PP.TAGS, "error", "missing"] and list(PP.DIALECT.kinds()) == [*PP.TAGS, "error", "missing"]
assert PP.HELPERS == () and PP.DIALECT.name() == "C Preprocessor" and PP.DIALECT.tag == "c_preprocessor"
assert PP.DIALECT.reads() == (C.DIALECT, CT.DIALECT, CPP.DIALECT, XT.DIALECT)  # C's and C++'s expressions and types
kinds = schemas(PP.DIALECT.union)
for dialect in (C.DIALECT, CPP.DIALECT):  # a condition's term, in C's and C++'s (first, as its install ran first)
    assert "definedexpression" in union(dialect.union)
    assert schemas(dialect.union)["definedexpression"] is kinds["definedexpression"] and dialect.kind_named("definedexpression") is PP.DIALECT.kinds()["definedexpression"]
for dialect in (CT.DIALECT, XT.DIALECT):  # an enumeration's items, in C Types' and Cpp Types'
    assert union(dialect.union)[:9] == [*PP.DIRECTIVES, "comment"] and all(schemas(dialect.union)[tag] is kinds[tag] for tag in PP.DIRECTIVES)
assert "comment" not in union(C.DIALECT.union) and "includedirective" not in union(CPP.DIALECT.union)  # no expression holds one
Holder = store.registered("Expressions.C.Preprocessor.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is store.registered(PP.DIALECT.union) and list(Holder.adjacencies) == ["abi"]
with store.transaction():
    holder = store.Expressions.C.Preprocessor.Expression(lambda e: e.root(PP.DIALECT.spec(if_)).abi([AMD64_]))
    linked = store.Schemas.OfExpression(lambda s: s.name("cpd.if").expression(lambda a: a.insert(lambda e: e.expression(holder))))
assert Validators.Validate(store)(Holder, holder) == [] and PP.DIALECT.validate(holder) == []
assert F.dialect_of(F.term(holder)) == PP.DIALECT.union and F.kind_of(F.term(holder))[0] == "ifdirective"
assert F.same(JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)).root, holder.root)
assert [F.same(entry.expression, holder) for entry in Builders.OfExpression.Builder(linked, False, store).expression] == [True]
with raises(Exception, match="'binaryexpression' is not a member of a union value"):  # its root holds its own terms
    with store.transaction():
        store.Expressions.C.Preprocessor.Expression(lambda e: e.root(PP.DIALECT.spec(C.binary("+", C.identifier("x"), 1))))


<a id="CPD-02"></a>
## CPD-02 · Its kinds: mbse-programs' properties in its order, a condition, a macro and the items any dialect's terms, a branch after another its own, a name C's identifier

In [ ]:
props = lambda name: list(store.registered(f"Expressions.C.Preprocessor.{name}").properties)  # noqa: E731
assert props("IncludeDirective") == ["directive", "path", "system", "macro"]
assert props("DefineDirective") == ["name", "function_like", "parameters", "variadic", "replacement"]
assert props("IfDirective") == props("ElifDirective") == ["condition", "items", "alternative"]
assert props("IfdefDirective") == props("ElifdefDirective") == ["negated", "name", "items", "alternative"]
assert props("ElseDirective") == ["items"] and props("OtherDirective") == ["directive", "text"]
assert props("DefinedExpression") == ["name"] and props("Comment") == ["block", "text", "trailing"]
prop = lambda name, member: store.registered(f"Expressions.C.Preprocessor.{name}").properties[member].schema  # noqa: E731
Term, Identifier = store.registered("Expressions.Framework.Term"), store.registered("Expressions.C.Expressions.Identifier")
assert prop("IfDirective", "condition") is prop("IncludeDirective", "macro") is prop("IfDirective", "items").element is Term
assert prop("IfDirective", "alternative") is prop("ElifdefDirective", "alternative") is store.registered(PP.DIALECT.union)
assert prop("DefinedExpression", "name") is prop("IfdefDirective", "name") is prop("DefineDirective", "parameters").element is Identifier
# As forms: a directive's condition and its branch after it are slots, its items its arguments after them
made = F.form(term(if_))
assert made.kind == "ifdirective" and [F.kind_of(a)[0] for a in made.arguments] == ["binaryexpression", "elifdefdirective", "enumerator"]
assert [F.dialect_of(a) for a in made.arguments] == [C.DIALECT.union, PP.DIALECT.union, CT.DIALECT.union]
assert F.same(term(PP.DIALECT.make(made)), term(if_))


<a id="CPD-03"></a>
## CPD-03 · Its checks: what each kind requires, an include's path or macro and its directive, a branch after another, an enumeration's items, a name's spelling, with mbse-programs' messages

In [ ]:
for spec, problems in [
    (lambda u: u.includedirective(lambda d: d), ["an IncludeDirective has a path or a macro"]),
    (lambda u: u.includedirective(lambda d: d.path("a.h").macro(C.identifier("H"))), ["an IncludeDirective has a path or a macro"]),
    (lambda u: u.includedirective(lambda d: d.macro(C.identifier("H"))), []),  # a macro's name, bound within it
    (lambda u: u.includedirective(lambda d: d.directive("require").path("a.h")), ["IncludeDirective.directive cannot be 'require'"]),
    (lambda u: u.includedirective(lambda d: d.directive("include_next").path("a.h")), []),
    (lambda u: u.definedirective(lambda d: d), ["a definedirective needs a name"]),
    (lambda u: u.definedirective(lambda d: d.name(spelled("F")).function_like(True).parameters([spelled("1a")])), ["'1a' is not an identifier"]),
    (lambda u: u.ifdirective(lambda d: d), ["an ifdirective needs a condition"]),
    (lambda u: u.ifdefdirective(lambda d: d), ["an ifdefdirective needs a name"]),
    (lambda u: u.elifdirective(lambda d: d), ["an elifdirective needs a condition"]),
    (lambda u: u.elifdefdirective(lambda d: d.name(spelled("2x"))), ["'2x' is not an identifier"]),
    (lambda u: u.definedexpression(lambda d: d), ["a definedexpression needs a name"]),
    (lambda u: u.ifdefdirective(lambda d: d.name(spelled("A")).alternative(lambda a: a.ifdefdirective(lambda e: e.name(spelled("B"))))),
     ["an IfdefDirective's alternative is an elifdirective, an elifdefdirective or an elsedirective"]),
    (lambda u: u.ifdirective(lambda d: d.condition(C.literal(1)).items([C.literal(2)])), ["an IfDirective's items are enumerators, directives and comments"]),
    (lambda u: u.elsedirective(lambda d: d.items([own(comment(" c")), enumerator("A", XT.DIALECT)])), []),  # any language's, from its holder
    (lambda u: u.otherdirective(lambda d: d), []), (lambda u: u.comment(lambda c: c), []),  # the null directive, an empty comment
]:
    assert validate(spec) == problems, (problems, validate(spec))
assert validate(lambda u: u.ifdirective(lambda d: d.condition(lambda c: c.c(lambda v: v.literal(lambda li: li.value(lambda n: n.as_scalar(1)).suffix("q")))))) == [
    "condition: a literal's suffix cannot be 'q'"]  # a C condition's own checks


<a id="CPD-04"></a>
## CPD-04 · What it binds: a conditional directive's macros' names, within its condition alone; defined's name is a macro's, not a reference; nothing refines Let

In [ ]:
assert validate(lambda u: u.ifdirective(lambda d: d.condition(C.binary(">", C.identifier("X"), 1)))) == []  # X names a macro
assert validate(lambda u: u.ifdirective(lambda d: d.condition(CPP.binary(">", CPP.identifier("X"), 1)))) == []  # C++'s too
assert PP.macros(term(condition, C.DIALECT)) == {"X"} and PP.macros(None) == set()
assert SY.free(term(if_), PP.DIALECT) == set() and SY.free(term(condition, C.DIALECT), C.DIALECT) == {"X"}  # bound by its directive alone
assert C.DIALECT.validate(term(defined("X"), C.DIALECT)) == [] and SY.free(term(defined("X"), C.DIALECT), C.DIALECT) == set()
valued = lambda u: u.ifdirective(lambda d: d.condition(C.identifier("X")).items([enumerator("A", CT.DIALECT)]).alternative(  # noqa: E731
    else_([CT.DIALECT.spec(lambda e: e.enumerator(lambda r: r.name(spelled("B")).value(C.identifier("X"))))])))
assert validate(valued) == ["alternative: argument 0: value: variable 'X' is not bound"]  # within its condition alone
assert not any(F.is_binder(b.schema) for b in store.registered(PP.DIALECT.union).branches)


<a id="CPD-05"></a>
## CPD-05 · Among the family's: defined(name) in C's and C++'s conditions, the directives and comments among C's and C++'s enumerators, each validated and printed from its family's holders

In [ ]:
# defined(name), a C term or a C++ term, within an expression
for dialect, printer in [(C.DIALECT, CText.ToText), (CPP.DIALECT, XText.ToText)]:
    either = dialect.spec(lambda u: u.binaryexpression(lambda b: b.left(defined("A", dialect)).operator("||").right(defined("B", dialect))))
    assert dialect.validate(term(either, dialect)) == [] and printer(term(either, dialect)) == "defined(A) || defined(B)"
    assert F.kind_of(term(defined("A", dialect), dialect))[0] == "definedexpression"
# Directives and comments among an enumeration's enumerators, in C's and C++'s, from C's and C++'s holders
def enumeration(types, items):
    return types.spec(lambda t: t.typeid(lambda i: i.specifiers([lambda s: s.enumspecifier(
        lambda e: e.key("enum").name(lambda n: n.identifier(spelled("E"))).body(True).enumerators(items))])))


for types, family, printer in [(CT.DIALECT, C, CText.ToText), (XT.DIALECT, CPP, XText.ToText)]:
    body = enumeration(types, [enumerator("A", types), types.spec(lambda u: u.ifdirective(lambda d: d.condition(
        family.binary(">", family.identifier("N"), 1)).items([enumerator("B", types)]).alternative(lambda a: a.elsedirective(
            lambda e: e.items([enumerator("C", types)]))))), types.spec(comment(" end"))])
    sized = family.DIALECT.spec(lambda u: u.sizeofexpression(lambda s: s.operand(body)))
    assert family.DIALECT.validate(term(sized, family.DIALECT)) == []
    assert printer(term(sized, family.DIALECT)) == "sizeof(enum E {\n    A\n#if N > 1\n    B,\n#else\n    C,\n#endif\n    // end\n})"
# A directive held from its own holder reads both languages: a C condition, a C++ item
assert validate(lambda u: u.ifdirective(lambda d: d.condition(condition).items([enumerator("A", XT.DIALECT)]))) == []


<a id="CPD-06"></a>
## CPD-06 · Invalid syntax: error and missing among its terms, validated and printed as written, an error with no text refused by name

In [ ]:
skipped, expected = lambda u: u.error(lambda r: r.text("#if (")), lambda u: u.missing(lambda r: r.kind("condition"))
assert validate(skipped) == ["invalid syntax '#if ('"] and validate(expected) == ["missing condition"]
assert text(skipped) == "#if (" and text(expected) == ""
assert validate(lambda u: u.ifdirective(lambda d: d.condition(lambda c: c.c(lambda v: v.missing(lambda m: m.kind("expression")))))) == [
    "condition: missing expression"]
with raises(ValueError, match="C Preprocessor error has no text to print"):
    text(lambda u: u.error(lambda r: r))


<a id="CPD-07"></a>
## CPD-07 · Its editions: C's three and C++'s three, each by its designation and common name, elifdef C23's and C++23's

In [ ]:
pre = store.Instances.Expressions.C.Preprocessor
assert [FS.tags(s) for s in FS.standards(pre).values()] == [
    ("ISO/IEC 9899:2011", "C11"), ("ISO/IEC 9899:2018", "C17"), ("ISO/IEC 9899:2024", "C23"), ("ISO/IEC 14882:2017", "C++17"),
    ("ISO/IEC 14882:2020", "C++20"), ("ISO/IEC 14882:2024", "C++23")] and PP.EDITIONS == ("C11", "C17", "C23", "C++17", "C++20", "C++23")
assert PP.SINCE == {"elifdefdirective": ("C23", "C++23")}
kinds = schemas(PP.DIALECT.union)
assert {tag: tuple(FS.tags(s)[-1] for s in FS.of(pre, kinds[tag]).values()) for tag in kinds} == {
    **{tag: PP.EDITIONS for tag in PP.TAGS}, "elifdefdirective": ("C23", "C++23"), "error": (), "missing": ()}
assert store.transaction(CPreprocessor.define)[0]["Cpp23"] is FS.find(pre, "C++23")  # run again, it writes nothing
assert Validators.Validate(store)(store.registered("Expressions.Framework.Standard"), FS.find(pre, "C11")) == []


<a id="CPD-08"></a>
## CPD-08 · Printing: each kind as mbse-programs' printer writes it, a directive at the start of its line, a conditional one's branches on lines of their own

In [ ]:
for spec, source in [
    (lambda u: u.includedirective(lambda d: d.path("a.h")), '#include "a.h"'),
    (lambda u: u.includedirective(lambda d: d.path("stdio.h").system(True)), "#include <stdio.h>"),
    (lambda u: u.includedirective(lambda d: d.directive("include_next").macro(C.identifier("HEADER"))), "#include_next HEADER"),
    (lambda u: u.definedirective(lambda d: d.name(spelled("N"))), "#define N"),
    (lambda u: u.definedirective(lambda d: d.name(spelled("N")).replacement("8")), "#define N 8"),
    (lambda u: u.definedirective(lambda d: d.name(spelled("MAX")).function_like(True).parameters([spelled("a"), spelled("b")])
        .replacement("((a) > (b) ? (a) : (b))")), "#define MAX(a, b) ((a) > (b) ? (a) : (b))"),
    (lambda u: u.definedirective(lambda d: d.name(spelled("LOG")).function_like(True).variadic(True).replacement("f(__VA_ARGS__)")),
     "#define LOG(...) f(__VA_ARGS__)"),
    (lambda u: u.ifdefdirective(lambda d: d.name(spelled("A"))), "#ifdef A\n#endif"),
    (lambda u: u.ifdefdirective(lambda d: d.negated(True).name(spelled("A")).items([enumerator("B")])), "#ifndef A\nB\n#endif"),
    (if_, "#if defined(X) && X > 1\nA\n#elifndef Z\nB\n#else\n// none\n#endif"),
    (lambda u: u.ifdirective(lambda d: d.condition(C.literal(0)).alternative(lambda a: a.elifdirective(lambda e: e.condition(
        CPP.binary("==", CPP.identifier("V"), 2)).alternative(lambda b: b.elifdefdirective(lambda f: f.name(spelled("W"))))))),
     "#if 0\n#elif V == 2\n#elifdef W\n#endif"),
    (lambda u: u.elsedirective(lambda e: e.items([enumerator("A")])), "#else\nA\n#endif"),
    (lambda u: u.ifdirective(lambda d: d.condition(C.literal(1)).items([enumerator("A"), own(lambda c: c.comment(
        lambda m: m.text(" a").trailing(True)))])), "#if 1\nA // a\n#endif"),
    (lambda u: u.otherdirective(lambda d: d.directive("pragma").text("once")), "#pragma once"),
    (lambda u: u.otherdirective(lambda d: d.directive("undef").text("N")), "#undef N"),
    (lambda u: u.otherdirective(lambda d: d), "#"),  # the null directive
    (lambda u: u.definedexpression(lambda d: d.name(spelled("X"))), "defined(X)"),
    (comment(" note"), "// note"), (comment(" a\n * b ", True), "/* a\n * b */"), (lambda u: u.comment(lambda c: c), "//"),
]:
    assert text(spec) == source, (source, text(spec))
with raises(ValueError, match="C Preprocessor includedirective has no path or macro to print"):
    text(lambda u: u.includedirective(lambda d: d))


<a id="CPD-09"></a>
## CPD-09 · Evaluation: none (Q11); each of its kinds is refused by name by its family's evaluators, wherever it stands

In [ ]:
for spec in [lambda u: u.includedirective(lambda d: d.path("a.h")), lambda u: u.definedirective(lambda d: d.name(spelled("N"))),
             if_, lambda u: u.otherdirective(lambda d: d.directive("pragma").text("once")), comment(" c"),
             lambda u: u.definedexpression(lambda d: d.name(spelled("X")))]:
    tag = F.kind_of(term(spec))[0]
    with raises(NotImplementedError, match=f"C Preprocessor {tag} is not evaluated"):
        CV.OfAny(term(spec))
with raises(NotImplementedError, match="C Preprocessor definedexpression is not evaluated"):  # within C's expression
    CV.OfAny(term(C.binary("&&", defined("A"), 1), C.DIALECT))
with raises(NotImplementedError, match="C Preprocessor definedexpression is not evaluated"):  # and C++'s
    XV.OfAny(term(CPP.binary("||", defined("A", CPP.DIALECT), 0), CPP.DIALECT))
